# Totimorphic lattice

This notebook constructs a totimorphic lattice by assembling cells from the [totimorphic cell notebook](./totimotphic_cell.ipynb).

In [1]:
# Project and symbolic tools
import heyoka as hy
import quivira as qv

# Numerics and visualization
import numpy as np
from _animation import animate_bar_model

# Building the lattice

The lattice is constructed, `n_x` and `n_y` are the number of cells in each direction of the grid.

In [2]:
model = qv.bar_joint_model()

# Number of cells in each direction
n_x = 4
n_y = 4

# Cell spacing
dx = 8
dy = 8

# Original cell geometry
base_point_pairs = [
    ((0, 0), (1, 4)),
    ((1, 4), (0, 8)),
    ((0, 8), (4, 7)),
    ((4, 7), (8, 8)),
    ((8, 8), (9, 4)),
    ((9, 4), (8, 0)),
    ((8, 0), (4, -1)),
    ((4, -1), (0, 0)),
]

# Keep track of already existing bars
existing_bars = set()

top_right_bar_index = None

for i in range(n_x):
    for j in range(n_y):

        offset_x = i * dx
        offset_y = j * dy

        for k, (start, end) in enumerate(base_point_pairs):

            new_start = (start[0] + offset_x, start[1] + offset_y)
            new_end = (end[0] + offset_x, end[1] + offset_y)

            bar_key = tuple(sorted((new_start, new_end)))

            if bar_key in existing_bars:
                continue

            # Identify bar 3 in the upper-right cell
            if i == n_x - 1 and j == n_y - 1 and k == 3:
                top_right_bar_index = len(model.bars)

            model.add_bar_from_point_pair(new_start, new_end)
            existing_bars.add(bar_key)

model.infer_joints()

model.add_support(bar_index=0, s=-0.5, dof_in_x=False, dof_in_y=False)

model.add_external_force(bar_index=top_right_bar_index, s=0.5, fx=hy.par[0], fy=hy.par[1])

In [3]:
ode_system, W = model.build_lagrange_eom(return_W=True)

hy.lagrangian took 0.364967 seconds
build_mass_matrix took 0.127923 seconds
build_constraint_jacobian took 0.067664 seconds
build_Jdqd took 0.224637 seconds
invert_matrix took 6.352721 seconds
force contribution took 0.092365 seconds
build W took 20.687118 seconds
build rhs took 0.049336 seconds
gaussian_elimination took 0.146170 seconds
acceleration correction took 11.549694 seconds
find_constrained_accelerations took 38.899808 seconds


In [ ]:
# import pickle
# from pathlib import Path

# output_path = Path("lagrange_eom_and_W.pkl")

# with output_path.open("wb") as f:
#     pickle.dump(
#         {"ode_system": ode_system, "W": W},
#         f,
#         protocol=pickle.HIGHEST_PROTOCOL,
#     )

NameError: name 'ode_system' is not defined

In [4]:
ta = hy.taylor_adaptive(ode_system, compact_mode=True)

In [5]:
state0 = model.get_initial_state()
model.check_constraint_fulfillment(state0)

In [6]:
ta.time = 0.0
ta.state[:] = state0

t_end = 10
tgrid = np.linspace(0.0, t_end, 401)
solution = ta.propagate_grid(tgrid)[-1]

In [7]:
# Use your existing 2D trajectory variable here:
animate_bar_model(model, solution, save_animation=True)